In [4]:
# Installation de PySpark (pour Google Colab)
!pip install pyspark --break-system-packages -q

print("✓ PySpark installé")

# Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
import os

# Créer la SparkSession
spark = SparkSession.builder \
    .appName("Oxford Pets - Traitement Images") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Ouvrir SparkUI en suivant le lien (Google Colab)
from google.colab import output
spark_ui_url = output.serve_kernel_port_as_window(4040, path='/jobs/index.html')

# Afficher les informations Spark
print("="*70)
print(f"✓ Spark {spark.version} démarré")
print(f"✓ Application: {spark.sparkContext.appName}")
print(f"✓ Spark UI URL: {spark_ui_url}")
print(f"✓ Cliquez sur le lien ci-dessus pour accéder à la SparkUI")
print("="*70)

✓ PySpark installé
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

✓ Spark 4.0.2 démarré
✓ Application: Oxford Pets - Traitement Images
✓ Spark UI URL: None
✓ Cliquez sur le lien ci-dessus pour accéder à la SparkUI


In [ ]:
# Importer la bibliothèque personnalisée
import sys
sys.path.insert(0, '/home/tarik/spark-workspace')

from lib import SparkJobMonitor, SparkHelper

# Initialiser le moniteur
monitor = SparkJobMonitor(spark)
helper = SparkHelper()

print("="*70)
print(f"✓ Spark {spark.version}")
print(f"✓ SparkJobMonitor chargé")
print(f"✓ SparkHelper chargé")
print(f"✓ Spark UI disponible sur: http://localhost:4040")
print("="*70)

✓ Spark 3.5.0
✓ SparkJobMonitor chargé
✓ Spark UI: http://localhost:4040


In [3]:
# ============================================
# TEST 1 : Comptage Simple
# ============================================

data = [
    ("Alice", 25, "IT", 50000),
    ("Bob", 30, "RH", 45000),
    ("Charlie", 35, "IT", 60000),
    ("David", 28, "Finance", 55000)
]

df = spark.createDataFrame(data, ["nom", "age", "departement", "salaire"])

result = monitor.execute_and_monitor(
    lambda: df.count(),
    "TEST 1: Comptage simple"
)

print(f"\nRésultat: {result} lignes")


🔵 TEST 1: Comptage simple
📌 Tracking ID: 68bb13ba


[Stage 0:>                                                          (0 + 4) / 4]


✅ SUCCESS
⏱️  Durée: 2.45s
📊 Spark Job ID(s): [1, 0]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

Résultat: 4 lignes


In [4]:
# ============================================
# TEST 2 : Agrégation
# ============================================

result = monitor.execute_and_monitor(
    lambda: df.groupBy("departement").count().show(),
    "TEST 2: Agrégation par département"
)


🔵 TEST 2: Agrégation par département
📌 Tracking ID: d3f142b0
+-----------+-----+
|departement|count|
+-----------+-----+
|         IT|    2|
|         RH|    1|
|    Finance|    1|
+-----------+-----+


✅ SUCCESS
⏱️  Durée: 1.65s
📊 Spark Job ID(s): [3, 2]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/


In [5]:
# ============================================
# TEST 3 : Gros Volume
# ============================================

result = monitor.execute_and_monitor(
    lambda: spark.range(0, 10_000_000).toDF("id").count(),
    "TEST 3: Génération de 10M lignes"
)

print(f"\nRésultat: {result:,} lignes")


🔵 TEST 3: Génération de 10M lignes
📌 Tracking ID: 07b732ff

✅ SUCCESS
⏱️  Durée: 0.24s
📊 Spark Job ID(s): [5, 4]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

Résultat: 10,000,000 lignes


In [6]:
# ============================================
# TEST 4 : Utilisation de SparkHelper
# ============================================

# Créer un DataFrame
df_test = spark.range(0, 1000).toDF("id")
from pyspark.sql.functions import rand

df_test = df_test.withColumn("valeur", (rand() * 100).cast("int"))

result = monitor.execute_and_monitor(
    lambda: helper.show_dataframe_info(df_test, "DataFrame Test"),
    "TEST 4: Analyse DataFrame"
)


🔵 TEST 4: Analyse DataFrame
📌 Tracking ID: e5e4826d

📊 Informations sur: DataFrame Test

🔢 Nombre de lignes: 1,000
📋 Nombre de colonnes: 2

📝 Schéma:
root
 |-- id: long (nullable = false)
 |-- valeur: integer (nullable = true)


👀 Aperçu des données:
+---+------+
|id |valeur|
+---+------+
|0  |3     |
|1  |3     |
|2  |3     |
|3  |44    |
|4  |3     |
+---+------+
only showing top 5 rows


📈 Statistiques:
+-------+-----------------+------------------+
|summary|               id|            valeur|
+-------+-----------------+------------------+
|  count|             1000|              1000|
|   mean|            499.5|            50.096|
| stddev|288.8194360957494|29.361963641911085|
|    min|                0|                 0|
|    max|              999|                99|
+-------+-----------------+------------------+


✅ SUCCESS
⏱️  Durée: 0.80s
📊 Spark Job ID(s): [10, 9, 8, 7, 6]
📦 Stages: 7 | Tasks: 19
🌐 Spark UI: http://localhost:4040/jobs/


In [7]:
# ============================================
# TEST 5 : Informations de Partitionnement
# ============================================

df_big = spark.range(0, 1_000_000).toDF("id")

result = monitor.execute_and_monitor(
    lambda: helper.get_partition_info(df_big),
    "TEST 5: Analyse du partitionnement"
)


🔵 TEST 5: Analyse du partitionnement
📌 Tracking ID: 530500ea

🗂️  Informations de Partitionnement

📦 Nombre de partitions: 4


[Stage 16:==============>                                           (1 + 3) / 4]


📊 Distribution des lignes par partition:
   Min: 250,000
   Max: 250,000
   Moyenne: 250,000

📋 Détail par partition:
   Partition 0: 250,000 lignes
   Partition 1: 250,000 lignes
   Partition 2: 250,000 lignes
   Partition 3: 250,000 lignes

✅ SUCCESS
⏱️  Durée: 1.49s
📊 Spark Job ID(s): [11]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/


In [8]:
# ============================================
# HISTORIQUE COMPLET
# ============================================

monitor.show_history()


📜 HISTORIQUE DES JOBS

📊 Statistiques globales:
   Total jobs: 5
   Réussis: 5
   Échoués: 0
   Durée totale: 6.65s


1. TEST 1: Comptage simple
   Tracking ID: 68bb13ba
   Spark Job IDs: [1, 0]
   Stages: 3 | Tasks: 9
   Durée: 2.45s
   Status: ✅ SUCCESS

2. TEST 2: Agrégation par département
   Tracking ID: d3f142b0
   Spark Job IDs: [3, 2]
   Stages: 3 | Tasks: 9
   Durée: 1.65s
   Status: ✅ SUCCESS

3. TEST 3: Génération de 10M lignes
   Tracking ID: 07b732ff
   Spark Job IDs: [5, 4]
   Stages: 3 | Tasks: 9
   Durée: 0.24s
   Status: ✅ SUCCESS

4. TEST 4: Analyse DataFrame
   Tracking ID: e5e4826d
   Spark Job IDs: [10, 9, 8, 7, 6]
   Stages: 7 | Tasks: 19
   Durée: 0.80s
   Status: ✅ SUCCESS

5. TEST 5: Analyse du partitionnement
   Tracking ID: 530500ea
   Spark Job IDs: [11]
   Stages: 1 | Tasks: 4
   Durée: 1.49s
   Status: ✅ SUCCESS


In [9]:
# ============================================
# Dernier Job Exécuté
# ============================================

last_job = monitor.get_last_job()

if last_job:
    print("\n📊 Détails du dernier job:")
    print(f"   Nom: {last_job['name']}")
    print(f"   Spark Job IDs: {last_job['spark_job_ids']}")
    print(f"   Durée: {last_job['duration']:.2f}s")
    print(f"   Status: {last_job['status']}")


📊 Détails du dernier job:
   Nom: TEST 5: Analyse du partitionnement
   Spark Job IDs: [11]
   Durée: 1.49s
   Status: ✅ SUCCESS
